In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
from maite_datasets.image_classification import MilitaryVehicles

from dataeval import Metadata
from dataeval.core import label_parity

In [3]:
train_ds = MilitaryVehicles("./data", image_set="train", download=True, lazy=True)
test_ds = MilitaryVehicles("./data", image_set="test", download=True, lazy=True)

# Get the labels from the collated dataset targets
train_labels = Metadata(train_ds).class_labels
test_labels = Metadata(test_ds).class_labels

In [4]:
results = label_parity(train_labels, test_labels, assume_independent=True)
print(
    f"The chi-squared value for the two label distributions is {results['chi_squared']:.1f}, "
    f"with p-value {results['p_value']:.1e} and score {results['score']:.2f}"
)

The chi-squared value for the two label distributions is 364.5, with p-value 3.5e-63 and score 0.19


In [5]:
controls = []
for seed in range(20):
    order = np.random.default_rng(seed).permutation(len(train_labels))
    half = len(order) // 2
    controls.append(label_parity(train_labels[order[:half]], train_labels[order[half:]], assume_independent=True))

print(f"Largest score over 20 random halves of the training set: {max(c['score'] for c in controls):.3f}")
print(f"Random halves with p < 0.05: {sum(c['p_value'] < 0.05 for c in controls)} of 20")

Largest score over 20 random halves of the training set: 0.037
Random halves with p < 0.05: 0 of 20


In [6]:
# TEST ASSERTION CELL ###
names = {name: index for index, name in train_ds.metadata["index2label"].items()}


def share(labels: np.ndarray, name: str) -> float:
    """Share of the labels that belong to the named class."""
    return float((labels == names[name]).mean())


assert len(train_labels) == 7823
assert len(test_labels) == 1621
assert results["p_value"] < 1e-50
assert not results["insufficient"]
assert results["test"] == "pearson"
assert abs(results["chi_squared"] - 364.5) < 0.1  # the text says the distributions differ
assert abs(results["score"] - 0.19) < 0.01  # and quotes 0.19
assert max(c["score"] for c in controls) < 0.04  # "below 0.04"
assert sum(c["p_value"] < 0.05 for c in controls) == 0  # "none of them is flagged"
assert results["score"] > 4.5 * max(c["score"] for c in controls)  # "about five times"
assert round(share(train_labels, "BM-30") * 100, 1) == 3.3
assert round(share(test_labels, "BM-30") * 100, 1) == 8.6
assert round(share(train_labels, "BMP-1") * 100, 1) == 5.0
assert round(share(test_labels, "BMP-1") * 100, 1) == 1.7